[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session4/BMEN35_Ex4_05_Optional_Image_generation_with_diffusion.ipynb)

# Optional walkthrough — Generate MNIST digits with a pretrained diffusion model

Keep the default CPU runtime in Colab. This walkthrough uses a pretrained model with about 4.3 MB of weights. We generate 28 × 28 grayscale images and follow one sample from random noise to the final result. Then we compare two choices for the number of sampling steps.

The idea is to see what happens **between** the initial noise and the final image. Run the sections in order so you can compare the intermediate images as well as the outputs. The first run installs the Diffusers package and downloads the model.

In [ ]:
%pip -q install diffusers

## 1. Load the network and the noise schedule

There are two main pieces.

1. The **U-Net** is the pretrained neural network. Given a noisy image and its time step, it estimates the noise in the current image. Its output has the same shape as the input image.
2. The **scheduler** specifies the noise levels and uses the network's estimate to calculate the next image during generation. The model was trained with a DDPM (Denoising Diffusion Probabilistic Models) schedule. We use the same noise levels with a DDIM (Denoising Diffusion Implicit Models) scheduler, which can take larger steps through that schedule. This keeps the number of network calls small.

The checkpoint was trained on [MNIST](https://huggingface.co/datasets/ylecun/mnist). We call it **unconditional** because the input is noise and a time step. There is no prompt or digit label. The image that emerges can represent **any digit**.

In [ ]:
import torch
from diffusers import DDIMScheduler, DDPMScheduler, UNet2DModel
from PIL import Image
from IPython.display import display

torch.set_num_threads(min(2, torch.get_num_threads()))  # Small batches run well with few CPU threads.

model_id = '1aurent/ddpm-mnist'
denoiser = UNet2DModel.from_pretrained(model_id, use_safetensors=True)
denoiser.to('cpu')
denoiser.eval()

# Use the checkpoint's training noise levels. Start both our experiments at the same time step.
training_config = DDPMScheduler.load_config(model_id)
scheduler = DDIMScheduler.from_config(training_config, timestep_spacing='trailing')

image_size = denoiser.config.sample_size
print('Image size:', image_size, '×', image_size)
print('Number of channels:', denoiser.config.in_channels)
print('Number of training time steps:', scheduler.config.num_train_timesteps)
print('Network prediction type:', scheduler.config.prediction_type)

## 2. Inspect the starting noise

An image tensor here has four axes in the order **images, channels, height, width**. A batch of four grayscale images therefore has shape `(4, 1, 28, 28)`. A grayscale image has one value per pixel rather than one for each colour channel.

We draw a Gaussian value for every pixel using a fixed seed. This is what the reverse process starts from. The display helper clips a copy for viewing. It uses the same brightness scale for the noise, the intermediate images and the final images. It leaves the tensors used by the model alone.

In [ ]:
def show_images(batch):
    tile_size = 84  # Enlarge each 28 × 28 image for display.
    gap = 4
    width = len(batch) * tile_size + (len(batch) - 1) * gap
    grid = Image.new('L', (width, tile_size), color=140)

    for index, image in enumerate(batch):
        pixels = image[0].detach().cpu().clamp(-1, 1)
        pixels = ((pixels + 1) * 127.5).round().to(torch.uint8).numpy()
        tile = Image.fromarray(pixels).resize(
            (tile_size, tile_size), Image.Resampling.NEAREST
        )
        grid.paste(tile, (index * (tile_size + gap), 0))

    display(grid)


seed = 7
rng = torch.Generator().manual_seed(seed)
initial_noise = torch.randn((4, 1, image_size, image_size), generator=rng)

print('Starting shape:', tuple(initial_noise.shape))
show_images(initial_noise)

## 3. Follow the denoising loop

At each step, the U-Net sees the current image and the current noise level. It predicts the noise, and the scheduler uses that prediction to compute an image at a lower noise level. The next iteration takes that updated image as input. The estimate is different each time because the current image and time step have changed.

`scheduler.timesteps` gives the time indices in reverse order. The original training schedule has 1,000 time indices. Choosing 40 sampling steps tells DDIM to visit 40 of them. It does not change the weights. In this notebook DDIM uses its deterministic setting, so for a given number of steps the next image is determined by the current one. A different starting noise image can produce a different digit.

In the loop, look for **two distinct assignments**. One stores the predicted noise and the other updates the current image. We save a few stages for display.

In [ ]:
def generate_digits(starting_noise, steps):
    scheduler.set_timesteps(steps)  # Choose which time indices to visit.
    current = starting_noise.clone()
    saved_stages = {}

    with torch.inference_mode():
        for update, time_step in enumerate(scheduler.timesteps, start=1):
            predicted_noise = denoiser(current, time_step).sample
            next_image = scheduler.step(predicted_noise, time_step, current)
            current = next_image.prev_sample

            if update in [steps // 4, steps // 2, steps]:
                saved_stages[update] = current.clone()

    return current, saved_stages


reference_steps = 40
digits_40, stages_40 = generate_digits(initial_noise, reference_steps)

Each tile below follows the **same** starting image. The first tile is pure noise. The rest show the result after 10, 20 and 40 updates. Use the next cell to see the four final images side by side.

**Pause.** When do you first see a stroke you recognise? Is it in the same place in the final image? Look for evidence in a particular tile before deciding when the digit appeared.

In [ ]:
first_image_stages = [initial_noise[0]]
for update, updated_images in stages_40.items():
    first_image_stages.append(updated_images[0])

print('Left to right: starting noise, then updates', list(stages_40))
show_images(first_image_stages)

print('Four final images after 40 sampling steps')
show_images(digits_40)

## 4. Change only the number of sampling steps

The next cell uses **the same four starting noise images** and the same pretrained weights. With 10 steps, the scheduler visits fewer time indices, so the network is called 10 times instead of 40. Compare the outputs column by column. Look for differences in the shape, the thickness of a stroke or how easily you recognise a digit.

A different number of steps changes the path through the noise schedule, so an image can turn into a different digit even from the same starting noise. Use the stage-by-stage images above to explain why this can happen.

In [ ]:
quick_steps = 10
digits_10, _ = generate_digits(initial_noise, quick_steps)

print('10 sampling steps')
show_images(digits_10)
print('40 sampling steps, same starting noise for each column')
show_images(digits_40)

## 5. Add noise to a finished digit

You can also run the process in the forward direction. We take one finished sample from step 3 and use it as a clean starting point $x_0$. At training time a real MNIST image would play that role. To illustrate the noise schedule we use the same draw $\epsilon$ in all three columns and change the time index $t$.

$$
x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon
$$

The schedule's $\bar{\alpha}_t$ determines how much weight the original image retains. As it gets smaller, the noise term gets a greater weight. A training example can be made at any chosen time step using this formula without simulating all the earlier steps. The network learns to estimate $\epsilon$ because that is the noise we drew.

In [ ]:
clean_example = digits_40[0:1]
time_indices = torch.tensor([100, 450, 900])
clean_copies = clean_example.repeat(len(time_indices), 1, 1, 1)

noise_rng = torch.Generator().manual_seed(23)
one_noise_draw = torch.randn(clean_example.shape, generator=noise_rng)
noise_copies = one_noise_draw.repeat(len(time_indices), 1, 1, 1)

noisy_copies = scheduler.add_noise(clean_copies, noise_copies, time_indices)
print('Left to right: finished sample, then time indices', time_indices.tolist())
show_images(torch.cat((clean_example, noisy_copies), dim=0))

## 6. Try a second starting point

Change `seed` in section 2, then rerun sections 2–4. Compare both sets of results after 10 and 40 steps. Which differences seem to repeat across seeds? Which appeared only in the first run?

In TinyStories, each step chooses a new token using the context so far. Here, each step updates all the pixels in the current image. In both examples you can save intermediate states to see how a final output takes shape.

[MNIST checkpoint and model card](https://huggingface.co/1aurent/ddpm-mnist) · [DDPM paper](https://arxiv.org/abs/2006.11239) · [DDIM paper](https://arxiv.org/abs/2010.02502).